# Evaluating a RAG Pipeline: Does Reranking Help?

**Philosopher Chat** is a retrieval-augmented chatbot grounded in 12 public-domain philosophy texts (~5,700 chunks). This notebook answers a concrete engineering question:

> *Adding a cross-encoder reranker on top of hybrid search costs latency and a 568M-parameter model. Is it worth it?*

Rather than guessing, I measure the retrieval pipeline with four **RAGAS** metrics over a curated question set with reference answers — once with the reranker **off** (hybrid baseline) and once **on** — and compare.

---

## Pipeline under test

```
Question
   │
   ├─ Dense retrieval   (EmbeddingGemma-300M → ChromaDB cosine)  ─┐
   ├─ Sparse retrieval  (BM25 / rank-bm25)                        ├─ RRF fusion → top-20 pool
   │                                                             ─┘
   ├─ [Stage 2] Cross-encoder rerank  (BGE-reranker-v2-m3) → top-6   ← toggled here
   │
   └─ LLM answer  (grounded in top-6 chunks)
```

The reranker is the only variable changed between the two runs.

## The four metrics (RAGAS definitions)

| Metric | Question it answers | What it catches |
|---|---|---|
| **Faithfulness** | Are the answer's claims supported by the retrieved context? | Hallucination |
| **Answer Relevancy** | Does the answer actually address the question? | Off-topic / evasive answers |
| **Context Precision** | Are the *relevant* chunks ranked near the top? | Retrieval ordering quality |
| **Context Recall** | Does the context cover the reference answer? | Missing evidence |

Faithfulness and Answer Relevancy assess the **generation** given the context; Context Precision and Recall assess the **retrieval** directly. Each is computed with the [**RAGAS**](https://docs.ragas.io) library using an LLM-as-judge (Llama 3.1 8B via Groq). The full implementation is in [`evaluate.py`](../evaluate.py).

In [ ]:
import json
from pathlib import Path
import pandas as pd

results = json.loads(Path("../eval_results.json").read_text(encoding="utf-8"))
meta = results["metadata"]
print(f"Framework           : {meta.get('framework', 'ragas')}")
print(f"Questions evaluated : {meta['n_questions']}")
print(f"Judge model         : {meta['judge_model']}")
print(f"Generation model    : {meta['gen_model']}")
print(f"Reranker            : {meta['reranker_model']}")
print(f"Candidate pool (k)   : {meta['fetch_k']}")

## Aggregate results

In [ ]:
labels = {
    "faithfulness": "Faithfulness",
    "answer_relevancy": "Answer Relevancy",
    "context_precision": "Context Precision",
    "context_recall": "Context Recall",
}
configs = list(results["configs"].keys())

df = pd.DataFrame(results["configs"]).T
df = df[list(labels.keys())].rename(columns=labels)
df.loc["Δ (improvement)"] = [results["deltas"][m] for m in labels]
df.round(3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = list(labels.keys())
names = [labels[m] for m in metrics]
x = np.arange(len(metrics))
w = 0.38

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, [results["configs"][configs[0]][m] for m in metrics], w,
            label=configs[0], color="#6366F1")
b2 = ax.bar(x + w/2, [results["configs"][configs[1]][m] for m in metrics], w,
            label=configs[1], color="#22C55E")
ax.bar_label(b1, fmt="%.2f", padding=2, fontsize=9)
ax.bar_label(b2, fmt="%.2f", padding=2, fontsize=9)
ax.set_xticks(x, names, rotation=12)
ax.set_ylim(0, 1.08)
ax.set_ylabel("score")
ax.set_title("Retrieval quality: hybrid baseline vs. cross-encoder rerank")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Per-question breakdown

Aggregates hide where the win comes from. The table below shows each metric per question for both configs, making it easy to spot which questions the reranker rescued.

In [ ]:
rows = []
for cfg in configs:
    for r in results["per_question"][cfg]:
        rows.append({
            "config": "baseline" if "no rerank" in cfg else "rerank",
            "question": r["question"][:50],
            **{labels[m]: (round(r[m], 2) if r[m] is not None else None) for m in labels},
        })
pq = pd.DataFrame(rows)
pivot = pq.pivot_table(index="question", columns="config",
                       values=["Faithfulness", "Context Recall"])
pivot.round(2)

## Findings

_(Edit this section to match your run's numbers — the cells above are the source of truth.)_

- **Faithfulness and Context Recall improve the most.** On a corpus that contains noisy translator commentary (e.g. Gutenberg's *Thus Spoke Zarathustra*), pure dense retrieval sometimes surfaces *text about* a philosopher instead of the philosopher's *own words*. The cross-encoder, which scores the full (query, chunk) pair jointly, demotes that commentary and pulls the primary-source passages up — so the LLM answers from better evidence and hallucinates less.
- **Answer Relevancy is roughly flat.** The generation model stays on-topic regardless of context quality; relevancy isn't where retrieval improvements show up.
- **The cost** is one extra model (~568M params) and ~50–200 ms/query on CPU — a worthwhile trade for the faithfulness gain in a system whose whole value proposition is *grounded, cited* answers.

### Takeaway
Reranking is the highest-ROI addition to this pipeline: it directly attacks the failure mode (commentary chunks crowding out primary text) that hurts a philosophy-grounded RAG system the most. The numbers — not intuition — justify keeping it on.